In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import signal
from ipywidgets import IntSlider, Output, RadioButtons, HBox, VBox
from IPython.display import display, clear_output, HTML


# ============================================================
# 1. Signal Preparation
# ============================================================

fs = 8000
duration = 1.0

t = np.linspace(
    0,
    duration,
    int(fs * duration),
    endpoint=False
)

carrier = signal.chirp(
    t,
    f0=150,
    f1=600,
    t1=duration,
    method='quadratic',
    phi=-90,
    vertex_zero=True
)

noise = np.random.normal(
    0,
    0.1,
    t.shape
)

vocal_signal = carrier + noise


# ============================================================
# 2. STFT Parameters
# ============================================================

frame_len = 300
hop_size = frame_len // 2


# ============================================================
# 3. Window Function
# ============================================================

def get_window(window_name):

    if window_name == 'Rectangular':
        return np.ones(frame_len)

    elif window_name == 'Hamming':
        return np.hamming(frame_len)

    elif window_name == 'Von Hann':
        return np.hanning(frame_len)

    elif window_name == 'Bartlett':
        return np.bartlett(frame_len)


# ============================================================
# 4. Initial STFT — Hamming
# ============================================================

window = get_window('Hamming')

f, t_stft, Zxx = signal.stft(
    vocal_signal,
    fs=fs,
    window=window,
    nperseg=frame_len,
    noverlap=hop_size
)

num_frames = len(t_stft)


# ============================================================
# 5. Output Area
# ============================================================

plot_output = Output()


# ============================================================
# 6. Figure and Subplots
# ============================================================

fig, (ax1, ax2, ax3) = plt.subplots(
    3,
    1,
    figsize=(17.25, 8),
    gridspec_kw={
        'height_ratios': [2.5, 2, 2]
    }
)

plt.subplots_adjust(
    hspace=0.4,
    right=0.90
)


# ============================================================
# 7. Upper Plot — Complete Signal
# ============================================================

ax1.plot(
    t,
    vocal_signal,
    color='black',
    alpha=0.3,
    label='Full Signal'
)

ax1.set_title(
    'Synthesized Speech-like Signal with Sliding Window',
    fontsize=11
)

ax1.set_ylabel(
    'Amplitude',
    fontsize=10
)

ax1.set_xlim(
    0,
    duration
)

ax1.set_ylim(
    -1.5,
    1.5
)

ax1.grid(True)


# ============================================================
# 8. STFT Windows on Complete Signal
# ============================================================

window_colors = plt.cm.viridis(
    np.linspace(0, 1, num_frames)
)

for i in range(num_frames):

    center_sample = int(
        t_stft[i] * fs
    )

    start_sample = max(
        0,
        center_sample - frame_len // 2
    )

    end_sample = min(
        len(t),
        start_sample + frame_len
    )

    ax1.fill_between(
        t[start_sample:end_sample],
        -1.5,
        1.5,
        color=window_colors[i],
        alpha=0.2
    )


# ============================================================
# 9. Current Frame Indicator
# ============================================================

initial_start = t[0]

initial_end = t[
    min(frame_len, len(t) - 1)
]

highlight_frame = Rectangle(
    (
        initial_start,
        -1.5
    ),
    initial_end - initial_start,
    3.0,
    color='red',
    alpha=0.4,
    label='Current Frame'
)

ax1.add_patch(
    highlight_frame
)


# ============================================================
# 10. Middle Plot — Isolated Frame
# ============================================================

line_frame, = ax2.plot(
    [],
    [],
    color='black',
    label='Original'
)

line_windowed, = ax2.plot(
    [],
    [],
    color='red',
    label='Windowed'
)

ax2.set_title(
    'Isolated Frame (Time-Domain with Hamming Window)',
    fontsize=11
)

ax2.set_xlabel(
    'Sample',
    fontsize=10
)

ax2.set_ylabel(
    'Amplitude',
    fontsize=10
)

ax2.set_xlim(
    0,
    frame_len - 1
)

ax2.set_ylim(
    -1.5,
    1.5
)

ax2.grid(True)


# ============================================================
# 11. Legend Outside Middle Plot
# ============================================================

ax2.legend(
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    fontsize=9,
    frameon=True
)


# ============================================================
# 12. Lower Plot — Magnitude Spectrum
# ============================================================

line_spectrum, = ax3.plot(
    [],
    [],
    color='red'
)

ax3.set_title(
    'Magnitude Spectrum of Current Frame — Hamming Window',
    fontsize=11
)

ax3.set_xlabel(
    'Frequency (Hz)',
    fontsize=10
)

ax3.set_ylabel(
    'Magnitude',
    fontsize=10
)

ax3.set_xlim(
    0,
    fs / 2
)

ax3.set_ylim(
    0,  1)

ax3.grid(True)


# ============================================================
# 13. Update Function
# ============================================================

def update_frame(frame_index, window_name):

    current_window = get_window(
        window_name
    )

    f_current, t_current, Zxx_current = signal.stft(
        vocal_signal,
        fs=fs,
        window=current_window,
        nperseg=frame_len,
        noverlap=hop_size
    )

    center_sample = int(
        t_current[frame_index] * fs
    )

    start_sample = max(
        0,
        center_sample - frame_len // 2
    )

    end_sample = start_sample + frame_len

    if end_sample > len(t):

        end_sample = len(t)

        start_sample = max(
            0,
            end_sample - frame_len
        )

    x_start = t[start_sample]

    x_end = t[end_sample - 1]

    highlight_frame.set_x(
        x_start
    )

    highlight_frame.set_width(
        x_end - x_start
    )

    current_segment = vocal_signal[
        start_sample:end_sample
    ]

    if len(current_segment) < frame_len:

        current_segment = np.pad(
            current_segment,
            (
                0,
                frame_len - len(current_segment)
            ),
            mode='constant'
        )

    current_windowed = (
        current_segment * current_window
    )

    samples = np.arange(
        frame_len
    )

    line_frame.set_data(
        samples,
        current_segment
    )

    line_windowed.set_data(
        samples,
        current_windowed
    )

    current_spectrum = np.abs(
        Zxx_current[:, frame_index]
    )

    line_spectrum.set_data(
        f_current,
        current_spectrum
    )

    ax2.set_title(
        f'Isolated Frame (Time-Domain with {window_name} Window)',
        fontsize=11
    )

    ax3.set_title(
        f'Magnitude Spectrum of Current Frame — {window_name} Window',
        fontsize=11
    )


# ============================================================
# 14. Frame Slider
# ============================================================

slider = IntSlider(
    min=0,
    max=num_frames - 1,
    step=1,
    value=0,
    description='Frame:',
    continuous_update=True,
    style={
        'description_width': 'initial'
    },
    layout={
        'width': '500px'
    }
)


# ============================================================
# 15. Window Selection — Horizontal
# ============================================================

window_selector = RadioButtons(
    options=[
        'Rectangular',
        'Hamming',
        'Von Hann',
        'Bartlett'
    ],
    value='Hamming',
    description='Window:',
    disabled=False
)

window_selector.layout = {
    'width': 'auto'
}

window_selector.style = {
    'description_width': '60px'
}

window_selector.add_class(
    'horizontal-radio'
)


# ============================================================
# 16. Horizontal Radio Button CSS
# ============================================================

display(HTML("""
<style>

.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    gap: 12px !important;
    align-items: center !important;
}

.horizontal-radio .widget-radio-item {
    margin-right: 8px !important;
}

.horizontal-radio {
    width: auto !important;
}

</style>
"""))


# ============================================================
# 17. Slider Callback
# ============================================================

def slider_changed(change):

    if change['name'] != 'value':
        return

    update_frame(
        change['new'],
        window_selector.value
    )

    with plot_output:

        clear_output(
            wait=True
        )

        display(fig)


slider.observe(
    slider_changed,
    names='value'
)


# ============================================================
# 18. Window Callback
# ============================================================

def window_changed(change):

    if change['name'] != 'value':
        return

    update_frame(
        slider.value,
        change['new']
    )

    with plot_output:

        clear_output(
            wait=True
        )

        display(fig)


window_selector.observe(
    window_changed,
    names='value'
)


# ============================================================
# 19. Initial Display
# ============================================================

update_frame(
    0,
    'Hamming'
)

with plot_output:
    display(fig)

plt.close(fig)


# ============================================================
# 20. Controls
# ============================================================

window_row = HBox(
    [
        window_selector
    ]
)

controls = VBox(
    [
        slider,
        window_row
    ]
)


# ============================================================
# 21. Final Display
# ============================================================

display(
    VBox(
        [
            controls,
            plot_output
        ]
    )
)